In [ ]:
# Simulate hallucination annotations
# In production, these would come from expert annotators

print("Generating hallucination annotations...")
print("Note: In production, annotations come from expert review")
print()

# Simulate binary hallucination labels (0 = no hallucination, 1 = hallucination)
# Hallucination rates vary by model and complexity
np.random.seed(42)

hallucination_labels = []

for _, row in df_responses.iterrows():
    # Hallucination probability depends on complexity and model
    complexity = row['complexity']
    model = row['model']
    
    # Base hallucination rates
    base_rates = {
        'simple': 0.10,
        'intermediate': 0.25,
        'complex': 0.40
    }
    
    # Model-specific adjustments
    model_adjustments = {
        'gpt-4-turbo': 0.95,      # Slightly better
        'claude-3-sonnet': 1.00,  # Baseline
        'gemini-1.5-pro': 1.05    # Slightly worse
    }
    
    # Calculate probability
    prob = base_rates.get(complexity, 0.25) * model_adjustments.get(model, 1.0)
    has_hallucination = np.random.random() < prob
    
    # Generate severity if hallucination present
    if has_hallucination:
        # Severity weighted towards lower values
        severity = np.random.choice([1, 2, 3, 4], p=[0.4, 0.3, 0.2, 0.1])
    else:
        severity = 0
    
    hallucination_labels.append({
        'query_id': row['query_id'],
        'model': model,
        'has_hallucination': int(has_hallucination),
        'severity': severity,
        'annotator': 'annotator_1'
    })

# Create annotations DataFrame
df_annotations = pd.DataFrame(hallucination_labels)

# Add annotations to responses
df_responses = df_responses.merge(
    df_annotations[['query_id', 'model', 'has_hallucination', 'severity']],
    on=['query_id', 'model'],
    how='left'
)

print(f"✓ Annotated {len(df_annotations)} responses")
print(f"\nHallucination Summary:")
print(f"  Total hallucinations: {df_annotations['has_hallucination'].sum()}")
print(f"  Hallucination rate: {df_annotations['has_hallucination'].mean():.2%}")
print(f"\nSeverity Distribution:")
print(df_annotations['severity'].value_counts().sort_index())

df_responses.head()

## 2. Hallucination Detection

Annotate responses for hallucinations using expert review and database validation.

**Hallucination Severity Scale:**
- **0**: No hallucination - Factually accurate
- **1**: Minor inaccuracy - Slightly imprecise but not misleading
- **2**: Moderate hallucination - Contains some incorrect information
- **3**: Severe hallucination - Substantially incorrect information
- **4**: Complete fabrication - Entirely invented information

**Detection Methods:**
1. Cross-reference with UniProt database
2. Verify against published literature (PubMed)
3. Expert review by domain specialists
4. Consensus labeling (2+ annotators)

In [ ]:
# Load benchmark results
loader = DataLoader()
results_path = Path('../data/llm_responses/benchmark_results_complete.csv')

if results_path.exists():
    df_responses = pd.read_csv(results_path)
    print(f"✓ Loaded {len(df_responses)} responses from benchmark")
else:
    print(f"⚠ Benchmark results not found at {results_path}")
    print("Creating mock dataset for demonstration...")
    
    # Create mock data
    models = ['gpt-4-turbo', 'claude-3-sonnet', 'gemini-1.5-pro']
    complexities = ['simple', 'intermediate', 'complex']
    
    mock_data = []
    for i in range(150):  # 50 queries × 3 models
        mock_data.append({
            'query_id': f"Q{(i // 3) + 1:03d}",
            'query_text': f"What is the function of protein {(i // 3) + 1}?",
            'complexity': complexities[(i // 3) % 3],
            'model': models[i % 3],
            'response_text': f"Mock response from {models[i % 3]} for query {(i // 3) + 1}",
            'status': 'success'
        })
    
    df_responses = pd.DataFrame(mock_data)
    print(f"✓ Created {len(df_responses)} mock responses")

# Filter successful responses
df_responses = df_responses[df_responses['status'] == 'success'].copy()

print(f"\nDataset Info:")
print(f"  Total responses: {len(df_responses)}")
print(f"  Unique queries: {df_responses['query_id'].nunique()}")
print(f"  Models: {df_responses['model'].unique().tolist()}")
print(f"  Complexity levels: {df_responses['complexity'].unique().tolist()}")

df_responses.head()

## 1. Load Benchmark Results

Load LLM responses from notebook 02 and prepare for hallucination detection.

# 03 - Hallucination Analysis

Comprehensive detection and categorization of hallucinations in LLM responses to proteomics queries.

**Objective:**  
Systematically identify and classify hallucinations across three LLM models (GPT-4 Turbo, Claude 3 Sonnet, Gemini Pro 1.5) using established proteomics databases as ground truth.

**Methods:**
- Cross-reference responses with UniProt, NCBI, and PDB databases
- Categorize hallucinations by severity (0-4 scale)
- Analyze patterns by query complexity and protein prevalence
- Calculate inter-rater reliability (Cohen's kappa)

**Study Information:**
- IRB Protocol: #2025-IRB-1101
- Date: November 2025
- Random Seed: 42

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Add source to path
sys.path.append('../src')

# Import project modules
from src.data_processing.loaders import DataLoader
from src.llm_eval.metrics import calculate_hallucination_metrics, severity_weighted_accuracy

# Set random seed
np.random.seed(42)

# Configure plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_style('whitegrid')
sns.set_palette("husl")

print("Hallucination analysis environment configured")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")